In [6]:
# imports
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import joblib

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

In [7]:
# loading trained MLP
class SimpleMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))

    def forward(self, x):

        return self.network(x)


model = SimpleMLP()

model.load_state_dict(torch.load("mlp_model.pth", map_location=torch.device("cpu")))

model.eval()

print("MLP Loaded")

MLP Loaded


In [8]:
# loading scaler
scaler = joblib.load("mlp_scaler.pkl")

print("Scaler Loaded")

Scaler Loaded


In [9]:
# creating dataset to feed to MLP from existing
df = pd.read_csv("test_predictions.csv")

mlp_df = df[["quality_score", "best_similarity", "margin", "label"]]

mlp_df.to_csv("degraded_mlp.csv", index=False)

print(mlp_df.shape)

mlp_df.head()

(540, 4)


,quality_score,best_similarity,margin,label
0,19.508541,0.750994,0.533056,1
1,15.662315,0.490444,0.290469,1
2,15.627174,0.587603,0.341852,1
3,20.108101,0.539806,0.306605,1
4,15.208915,0.700871,0.490324,1


In [10]:
# running MLP on degraded dataset
df = pd.read_csv("degraded_mlp.csv")

X = df[["quality_score", "best_similarity", "margin"]]

y_true = df["label"]

X_scaled = scaler.transform(X)

X_tensor = torch.tensor(X_scaled, dtype=torch.float32)

with torch.no_grad():

    outputs = model(X_tensor)

    probabilities = torch.softmax(outputs, dim=1)

    y_pred = torch.argmax(outputs, dim=1).numpy()

    df["mlp_decision"] = y_pred
    df.to_csv("degraded_mlp_predictions.csv", index=False)
    print("Saved degraded_mlp_predictions.csv")

Saved degraded_mlp_predictions.csv


In [11]:
# MLP metrics
accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred)

recall = recall_score(y_true, y_pred)

f1 = f1_score(y_true, y_pred)

print("===== MLP RESULTS =====\n")

print("Accuracy :", accuracy)

print("Precision:", precision)

print("Recall   :", recall)

print("F1 Score :", f1)

cm_mlp = confusion_matrix(y_true, y_pred)

print("\nConfusion Matrix")

print(cm_mlp)

print(classification_report(y_true, y_pred))

===== MLP RESULTS =====

Accuracy : 0.9388888888888889
Precision: 1.0
Recall   : 0.9375
F1 Score : 0.967741935483871

Confusion Matrix
[[ 12   0]
 [ 33 495]]
              precision    recall  f1-score   support

           0       0.27      1.00      0.42        12
           1       1.00      0.94      0.97       528

    accuracy                           0.94       540
   macro avg       0.63      0.97      0.69       540
weighted avg       0.98      0.94      0.96       540



In [12]:
# TRR FAR FRR TAR
TN, FP, FN, TP = cm_mlp.ravel()

TAR = TP / (TP + FN)

FRR = FN / (TP + FN)

TRR = TN / (TN + FP)

FAR = FP / (TN + FP)

print("\n===== VERIFICATION METRICS =====")

print("TAR:", TAR)

print("FRR:", FRR)

print("TRR:", TRR)

print("FAR:", FAR)


===== VERIFICATION METRICS =====
TAR: 0.9375
FRR: 0.0625
TRR: 1.0
FAR: 0.0


In [13]:
# Fixed threshold method
threshold = 0.6

threshold_pred = (df["best_similarity"] >= threshold).astype(int)

accuracy_th = accuracy_score(y_true, threshold_pred)

precision_th = precision_score(y_true, threshold_pred)

recall_th = recall_score(y_true, threshold_pred)

f1_th = f1_score(y_true, threshold_pred)

print("===== THRESHOLD RESULTS =====\n")

print("Accuracy :", accuracy_th)

print("Precision:", precision_th)

print("Recall   :", recall_th)

print("F1 Score :", f1_th)

cm_threshold = confusion_matrix(y_true, threshold_pred)

print("\nConfusion Matrix")

print(cm_threshold)

print(classification_report(y_true, threshold_pred))

===== THRESHOLD RESULTS =====

Accuracy : 0.6037037037037037
Precision: 1.0
Recall   : 0.5946969696969697
F1 Score : 0.7458432304038005

Confusion Matrix
[[ 12   0]
 [214 314]]
              precision    recall  f1-score   support

           0       0.05      1.00      0.10        12
           1       1.00      0.59      0.75       528

    accuracy                           0.60       540
   macro avg       0.53      0.80      0.42       540
weighted avg       0.98      0.60      0.73       540



In [14]:
# Fized threshold TRR TAR FRR FAR
TN, FP, FN, TP = cm_threshold.ravel()

TAR_th = TP / (TP + FN)

FRR_th = FN / (TP + FN)

TRR_th = TN / (TN + FP)

FAR_th = FP / (TN + FP)

print("\n===== THRESHOLD METRICS =====")

print("TAR:", TAR_th)

print("FRR:", FRR_th)

print("TRR:", TRR_th)

print("FAR:", FAR_th)


===== THRESHOLD METRICS =====
TAR: 0.5946969696969697
FRR: 0.4053030303030303
TRR: 1.0
FAR: 0.0


In [15]:
# final comparison table
svm_accuracy = 0.9740740740740741
svm_precision = 0.998062
svm_recall = 0.975379
svm_f1 = 0.986590

svm_tar = 0.9753787878787878
svm_frr = 0.02462121212121212
svm_trr = 0.9166666666666666
svm_far = 0.08333333333333333

comparison = pd.DataFrame(
    {
        "Method": [
            "Threshold",
            "SVM",
            "MLP"
        ],

        "Accuracy": [
            accuracy_th,
            svm_accuracy,
            accuracy
        ],

        "Precision": [
            precision_th,
            svm_precision,
            precision
        ],

        "Recall": [
            recall_th,
            svm_recall,
            recall
        ],

        "F1": [
            f1_th,
            svm_f1,
            f1
        ],

        "TAR": [
            TAR_th,
            svm_tar,
            TAR
        ],

        "FRR": [
            FRR_th,
            svm_frr,
            FRR
        ],

        "TRR": [
            TRR_th,
            svm_trr,
            TRR
        ],

        "FAR": [
            FAR_th,
            svm_far,
            FAR
        ]
    }
)

comparison

,Method,Accuracy,Precision,Recall,F1,TAR,FRR,TRR,FAR
0,Threshold,0.603704,1.000000,0.594697,0.745843,0.594697,0.405303,1.000000,0.000000
1,SVM,0.974074,0.998062,0.975379,0.986590,0.975379,0.024621,0.916667,0.083333
2,MLP,0.938889,1.000000,0.937500,0.967742,0.937500,0.062500,1.000000,0.000000
